In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
master = pd.read_csv(
    "../../data/raw/aircraft/MASTER.txt",
    low_memory=False
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [3]:
# ============================================================
# Create Working Copy
# ============================================================

aircraft_clean = master.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [4]:
# ============================================================
# Remove Empty Column
# ============================================================

aircraft_clean.drop(columns=["Unnamed: 34"], inplace=True)

print("Unnamed: 34 removed.")
print("Remaining Columns:", aircraft_clean.shape[1])

Unnamed: 34 removed.
Remaining Columns: 34


In [5]:
# ============================================================
# Replace Blank Strings with NaN
# ============================================================

aircraft_clean = aircraft_clean.replace(r'^\s*$', np.nan, regex=True)

print("Blank values converted to NaN.")

Blank values converted to NaN.


In [6]:
missing_summary = pd.DataFrame({
    "Missing Values": aircraft_clean.isnull().sum(),
    "Percentage (%)": (
        aircraft_clean.isnull().sum() / len(aircraft_clean) * 100
    ).round(2)
})

missing_summary.sort_values(
    "Missing Values",
    ascending=False
).head(15)

,Missing Values,Percentage (%)
OTHER NAMES(5),314018,99.87
OTHER NAMES(4),313900,99.84
OTHER NAMES(3),313494,99.71
FRACT OWNER,313325,99.65
OTHER NAMES(2),311601,99.10
STREET2,304976,97.00
KIT MFR,297457,94.61
KIT MODEL,297457,94.61
OTHER NAMES(1),283257,90.09
YEAR MFR,68261,21.71


In [7]:
# ============================================================
# Remove Administrative Columns
# ============================================================

columns_to_drop = [

    "NAME",
    "STREET",
    "STREET2",
    "CITY",
    "STATE",
    "ZIP CODE",
    "COUNTY",
    "COUNTRY",
    "REGION",

    "OTHER NAMES(1)",
    "OTHER NAMES(2)",
    "OTHER NAMES(3)",
    "OTHER NAMES(4)",
    "OTHER NAMES(5)",

    "FRACT OWNER"

]

aircraft_clean.drop(
    columns=columns_to_drop,
    inplace=True
)

print("Administrative columns removed.")
print(aircraft_clean.shape)

Administrative columns removed.
(314417, 19)


In [8]:
# ============================================================
# Clean Manufacturing Year
# ============================================================

aircraft_clean["YEAR MFR"] = pd.to_numeric(
    aircraft_clean["YEAR MFR"],
    errors="coerce"
)

aircraft_clean.loc[
    aircraft_clean["YEAR MFR"] == 0,
    "YEAR MFR"
] = np.nan

print("Manufacturing year cleaned.")

Manufacturing year cleaned.


In [9]:
aircraft_clean["YEAR MFR"].describe()

count    246006.000000
mean       1984.467363
std          24.154516
min        1909.000000
25%        1966.000000
50%        1979.000000
75%        2006.000000
max        2026.000000
Name: YEAR MFR, dtype: float64

In [10]:
# ============================================================
# Convert Date Columns
# ============================================================

date_columns = [

    "LAST ACTION DATE",
    "CERT ISSUE DATE",
    "AIR WORTH DATE",
    "EXPIRATION DATE"

]

for column in date_columns:

    aircraft_clean[column] = pd.to_datetime(

        aircraft_clean[column],

        format="%Y%m%d",

        errors="coerce"

    )

print("Date conversion completed.")

Date conversion completed.


In [11]:
aircraft_clean[date_columns].dtypes

LAST ACTION DATE    datetime64[ns]
CERT ISSUE DATE     datetime64[ns]
AIR WORTH DATE      datetime64[ns]
EXPIRATION DATE     datetime64[ns]
dtype: object

In [12]:
aircraft_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314417 entries, 0 to 314416
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   N-NUMBER          314417 non-null  object        
 1   SERIAL NUMBER     314417 non-null  object        
 2   MFR MDL CODE      314417 non-null  object        
 3   ENG MFR MDL       276838 non-null  object        
 4   YEAR MFR          246006 non-null  float64       
 5   TYPE REGISTRANT   313225 non-null  object        
 6   LAST ACTION DATE  314417 non-null  datetime64[ns]
 7   CERT ISSUE DATE   306526 non-null  datetime64[ns]
 8   CERTIFICATION     280514 non-null  object        
 9   TYPE AIRCRAFT     314417 non-null  object        
 10  TYPE ENGINE       314417 non-null  int64         
 11  STATUS CODE       314417 non-null  object        
 12  MODE S CODE       314417 non-null  int64         
 13  AIR WORTH DATE    272438 non-null  datetime64[ns]
 14  EXPI

In [13]:
print("="*60)
print("Cleaned Dataset Shape")
print("="*60)

print(f"Rows    : {aircraft_clean.shape[0]:,}")
print(f"Columns : {aircraft_clean.shape[1]}")

Cleaned Dataset Shape
Rows    : 314,417
Columns : 19


In [14]:
missing_summary = pd.DataFrame({

    "Missing Values": aircraft_clean.isnull().sum(),

    "Percentage (%)": (
        aircraft_clean.isnull().sum() /
        len(aircraft_clean) * 100
    ).round(2)

})

missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

,Missing Values,Percentage (%)
KIT MFR,297457,94.61
KIT MODEL,297457,94.61
YEAR MFR,68411,21.76
AIR WORTH DATE,41979,13.35
ENG MFR MDL,37579,11.95
CERTIFICATION,33903,10.78
EXPIRATION DATE,7923,2.52
CERT ISSUE DATE,7891,2.51
TYPE REGISTRANT,1192,0.38
N-NUMBER,0,0.00


In [17]:
aircraft_clean.drop(
    columns=["KIT MFR", " KIT MODEL"],
    inplace=True
)

In [18]:
output_path = "../../data/processed/aircraft/aircraft_metadata_processed.csv"

aircraft_clean.to_csv(
    output_path,
    index=False
)

print("Aircraft metadata processed dataset saved successfully.")

Aircraft metadata processed dataset saved successfully.
